# 🚀 Bit-MC-SSM: Scale-Up GPU Training & 2-bit Export
### 〜 15M〜30M 規模の本格学習と C++ Zero-GEMM エンジンへの即時エクスポート 〜

本ノートブックは、**1.58-bit 三値 LLM（Bit-MC-SSM）** を Google Colab の GPU（T4 / A100）を活用して 15M〜30M パラメータ規模で高速学習し、
**2-bit パックドバイナリ（`model_30m.bin`）** としてダウンロードして手元の PC（C++ エンジン）で動かすためのオールインワン環境です。

#### 🌟 特徴:
* **Mixed Precision (AMP / BF16):** GPU での学習を極限まで加速
* **Top-2 疎逆伝播 (Sparse Backprop):** 長文時系列の勾配計算・メモリ消費を大幅削減
* **ワンクリック 2-bit バイナリパッキング:** 学習完了後、PC の L3 キャッシュに収まる 2-bit バイナリを自動生成・ダウンロード

## 1. 依存ライブラリのインストール & GPU 検証

In [1]:
!pip install -q transformers datasets einops tiktoken

import os
import time
import math
import json
import struct
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2TokenizerFast
from datasets import load_dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Active Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

🚀 Active Device: cuda
   GPU Model: Tesla T4
   VRAM: 15.64 GB


## 2. モデル規模プリセットの設定
目的に合わせてモデルサイズを選択してください（Colab T4 では **medium-30M** が最もおすすめです）。

In [2]:
# 選択肢: 'small-15M', 'medium-30M', 'large-60M'
PRESET = 'medium-30M'

CONFIGS = {
    'small-15M': {
        'd_model': 256,
        'n_layers': 6,
        'd_state': 32,
        'batch_size': 32,
        'lr': 1.5e-3,
        'epochs': 8,
    },
    'medium-30M': {
        'd_model': 384,
        'n_layers': 8,
        'd_state': 32,
        'batch_size': 24,
        'lr': 1.0e-3,
        'epochs': 10,
    },
    'large-60M': {
        'd_model': 512,
        'n_layers': 8,
        'd_state': 64,
        'batch_size': 16,
        'lr': 8e-4,
        'epochs': 12,
    }
}

cfg = CONFIGS[PRESET]
SEQ_LEN = 128
SEGMENT_LEN = 32
TOP_K = 2
print(f"Selected Config [{PRESET}]: d_model={cfg['d_model']}, n_layers={cfg['n_layers']}, d_state={cfg['d_state']}")

Selected Config [medium-30M]: d_model=384, n_layers=8, d_state=32


## 3. Bit-MC-SSM アーキテクチャ定義 (1.58-bit STE ＋ 疎逆伝播)

In [3]:
class Quantize158(torch.autograd.Function):
    @staticmethod
    def forward(ctx, weight):
        gamma = weight.abs().mean().clamp(min=1e-5)
        w_scaled = weight / gamma
        w_ternary = torch.clamp(torch.round(w_scaled), -1.0, 1.0)
        return w_ternary * gamma

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output

class BitLinear158(nn.Linear):
    def forward(self, x):
        w_q = Quantize158.apply(self.weight)
        return F.linear(x, w_q, self.bias)

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x * norm * self.weight

class ShiftSSM(nn.Module):
    def __init__(self, d_model, d_state=32):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.in_proj = BitLinear158(d_model, 2 * d_model)
        self.conv1d = nn.Conv1d(d_model, d_model, kernel_size=4, padding=3, groups=d_model)
        self.b_proj = BitLinear158(d_model, d_state)
        self.c_proj = BitLinear158(d_model, d_state)
        self.decay_param = nn.Parameter(torch.tensor([-2.0] * d_state))
        self.out_proj = BitLinear158(d_model, d_model)

    def forward(self, x, cached_state=None):
        B, L, D = x.shape
        proj = self.in_proj(x)
        u, gate = proj.chunk(2, dim=-1)

        u_conv = self.conv1d(u.transpose(1, 2))[:, :, :L].transpose(1, 2)
        u_conv = F.silu(u_conv)

        B_t = self.b_proj(u_conv)
        C_t = self.c_proj(u_conv)
        decay = torch.sigmoid(self.decay_param)

        if cached_state is not None:
            h = cached_state
        else:
            h = torch.zeros(B, self.d_state, device=x.device, dtype=x.dtype)

        y_list = []
        u_scalar = u_conv.mean(dim=-1, keepdim=True)
        for t in range(L):
            h = decay * h + B_t[:, t, :] * u_scalar[:, t, :]
            y_t = (C_t[:, t, :] * h).sum(dim=-1, keepdim=True)
            y_list.append(y_t)

        y_ssm = torch.cat(y_list, dim=-1).unsqueeze(-1).expand(-1, -1, D)
        y = (u_conv + y_ssm) * F.silu(gate)
        return self.out_proj(y), h

class BitMCSSMBlock(nn.Module):
    def __init__(self, d_model, d_state=32):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.ssm = ShiftSSM(d_model, d_state)
        self.norm2 = RMSNorm(d_model)
        self.ffn_in = BitLinear158(d_model, d_model * 4)
        self.ffn_out = BitLinear158(d_model * 2, d_model)

    def forward(self, x, cached_state=None):
        ssm_out, next_state = self.ssm(self.norm1(x), cached_state)
        x = x + ssm_out
        ffn_p = self.ffn_in(self.norm2(x))
        f1, f2 = ffn_p.chunk(2, dim=-1)
        x = x + self.ffn_out(F.silu(f1) * f2)
        return x, next_state

class BitMCSSMModel(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, d_state=32, segment_len=32, top_k=2):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.n_layers = n_layers
        self.segment_len = segment_len
        self.top_k = top_k

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([BitMCSSMBlock(d_model, d_state) for _ in range(n_layers)])
        self.norm_f = RMSNorm(d_model)
        self.head = BitLinear158(d_model, vocab_size, bias=False)

    def forward_sparse_bp(self, input_ids):
        B, L = input_ids.shape
        num_segments = (L + self.segment_len - 1) // self.segment_len
        x = self.embedding(input_ids)

        # Top-k segment selection for sparse backpropagation
        if self.training and num_segments > self.top_k:
            perm = torch.randperm(num_segments, device=input_ids.device)
            bp_indices = set(perm[:self.top_k].tolist())
        else:
            bp_indices = set(range(num_segments))

        cached_states = [None] * self.n_layers
        segment_outputs = []

        for seg_idx in range(num_segments):
            st = seg_idx * self.segment_len
            ed = min(st + self.segment_len, L)
            seg_x = x[:, st:ed, :]

            if self.training and seg_idx not in bp_indices:
                seg_cached = [s.detach() if s is not None else None for s in cached_states]
            else:
                seg_cached = cached_states

            for layer_idx, block in enumerate(self.blocks):
                seg_x, new_s = block(seg_x, seg_cached[layer_idx])
                cached_states[layer_idx] = new_s

            segment_outputs.append(seg_x)

        full_out = torch.cat(segment_outputs, dim=1)
        logits = self.head(self.norm_f(full_out))
        return logits

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=40, temperature=0.7, top_k=40):
        self.eval()
        generated = input_ids.clone()
        for _ in range(max_new_tokens):
            logits = self.forward_sparse_bp(generated)
            next_token_logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(next_token_logits, min(top_k, next_token_logits.size(-1)))
                next_token_logits[next_token_logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            generated = torch.cat([generated, next_token], dim=1)
        return generated

## 4. データセットの準備 (TinyStories ストリーミング)

In [4]:
print("📥 Loading GPT-2 Fast Tokenizer & TinyStories Dataset...")
tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

# TinyStories をストリーミング取得してトークン化
raw_dataset = load_dataset('roneneldan/TinyStories', split='train', streaming=True)

class StreamStoryDataset(Dataset):
    def __init__(self, raw_data, tokenizer, num_samples=15000, seq_len=128):
        self.samples = []
        print(f"   Tokenizing {num_samples} stories...")
        count = 0
        for item in raw_data:
            text = item['text'].strip()
            if len(text) < 50:
                continue
            toks = tokenizer.encode(text)
            if len(toks) >= seq_len + 1:
                self.samples.append(torch.tensor(toks[:seq_len + 1], dtype=torch.long))
                count += 1
                if count >= num_samples:
                    break
        print(f"✅ Successfully prepared {len(self.samples)} training sequences!")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        seq = self.samples[idx]
        return seq[:-1], seq[1:]

train_dataset = StreamStoryDataset(raw_dataset, tokenizer, num_samples=12000, seq_len=SEQ_LEN)
train_loader = DataLoader(train_dataset, batch_size=cfg['batch_size'], shuffle=True)

📥 Loading GPT-2 Fast Tokenizer & TinyStories Dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

   Tokenizing 12000 stories...


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1106 > 1024). Running this sequence through the model will result in indexing errors


✅ Successfully prepared 12000 training sequences!


## 5. GPU 高速学習 (AMP + Cosine Annealing Scheduler)

In [ ]:
model = BitMCSSMModel(
    vocab_size=tokenizer.vocab_size,
    d_model=cfg['d_model'],
    n_layers=cfg['n_layers'],
    d_state=cfg['d_state'],
    segment_len=SEGMENT_LEN,
    top_k=TOP_K
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"🧠 Model Parameters: {total_params / 1e6:.2f}M ({total_params:,} params)")

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['epochs'] * len(train_loader))
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

print("\n🔥 Training Started...")
start_train_time = time.time()

for epoch in range(1, cfg['epochs'] + 1):
    model.train()
    total_loss = 0.0
    epoch_start = time.time()

    for step, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            logits = model.forward_sparse_bp(inputs)
            loss = F.cross_entropy(logits.view(-1, tokenizer.vocab_size), targets.view(-1))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    ppl = math.exp(min(avg_loss, 10.0))
    print(f"Epoch [{epoch:02d}/{cfg['epochs']:02d}] | Loss: {avg_loss:.4f} | Perplexity: {ppl:.2f} | Time: {time.time() - epoch_start:.1f}s")

print(f"\n✅ Total Training Completed in {time.time() - start_train_time:.1f}s!")

🧠 Model Parameters: 49.46M (49,458,048 params)


/tmp/ipykernel_1848/3033469782.py:15: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))
/tmp/ipykernel_1848/3033469782.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):



🔥 Training Started...
Epoch [01/10] | Loss: 3.6764 | Perplexity: 39.50 | Time: 307.6s
Epoch [02/10] | Loss: 2.8491 | Perplexity: 17.27 | Time: 305.4s


## 6. 学習後テキスト生成テスト (GPU)

In [ ]:
prompts = [
    "Once upon a time, Lily saw a tiny",
    "The little dog loved to play in the",
    "One sunny morning, Tom found a magic"
]

print("=" * 70)
print("✨ Text Generation Verification:")
print("=" * 70)

for p in prompts:
    prompt_ids = torch.tensor([tokenizer.encode(p)], dtype=torch.long, device=device)
    out_ids = model.generate(prompt_ids, max_new_tokens=40, temperature=0.7)
    text = tokenizer.decode(out_ids[0].tolist(), skip_special_tokens=True)
    print(f"[Prompt]: {p}")
    print(f"[Output]: {text}")
    print("-" * 70)

## 7. 2-bit バイナリパッキング & ワンクリックダウンロード
学習した重みを 2-bit パックド形式（`model_30m.bin`）に圧縮し、Colab から直接ダウンロードします。

In [ ]:
def pack_ternary_weights(weight_tensor):
    gamma = weight_tensor.abs().mean().item()
    gamma = max(gamma, 1e-5)
    w_scaled = weight_tensor / gamma
    w_ternary = torch.clamp(torch.round(w_scaled), -1.0, 1.0).to(torch.int8)
    w_flat = w_ternary.cpu().numpy().flatten()
    n = len(w_flat)
    pad = (4 - (n % 4)) % 4
    if pad > 0:
        w_flat = np.pad(w_flat, (0, pad), mode='constant', constant_values=0)
    w_code = np.zeros_like(w_flat, dtype=np.uint8)
    w_code[w_flat == 0] = 0
    w_code[w_flat == 1] = 1
    w_code[w_flat == -1] = 2
    w_reshaped = w_code.reshape(-1, 4)
    packed = (w_reshaped[:, 0] | (w_reshaped[:, 1] << 2) | (w_reshaped[:, 2] << 4) | (w_reshaped[:, 3] << 6)).astype(np.uint8)
    return gamma, packed.tobytes()

bin_path = f"model_{PRESET}.bin"
print(f"📦 Exporting 2-bit Packed Binary to: {bin_path}...")

with open(bin_path, "wb") as f:
    # Header: Magic, Vocab, d_model, n_layers, d_state
    f.write(b"BSSM")
    f.write(struct.pack("<IIII", tokenizer.vocab_size, cfg['d_model'], cfg['n_layers'], cfg['d_state']))

    # Embeddings (FP32)
    emb_data = model.embedding.weight.detach().cpu().numpy().astype(np.float32).tobytes()
    f.write(emb_data)

    # Blocks
    for b in model.blocks:
        # norm1
        f.write(b.norm1.weight.detach().cpu().numpy().astype(np.float32).tobytes())
        # in_proj
        g, p = pack_ternary_weights(b.ssm.in_proj.weight.detach())
        f.write(struct.pack("<f", g)); f.write(p)
        # conv1d
        f.write(b.ssm.conv1d.weight.detach().cpu().numpy().astype(np.float32).tobytes())
        if b.ssm.conv1d.bias is not None:
            f.write(b.ssm.conv1d.bias.detach().cpu().numpy().astype(np.float32).tobytes())
        else:
            f.write(np.zeros(cfg['d_model'], dtype=np.float32).tobytes())
        # b_proj, c_proj
        g, p = pack_ternary_weights(b.ssm.b_proj.weight.detach())
        f.write(struct.pack("<f", g)); f.write(p)
        g, p = pack_ternary_weights(b.ssm.c_proj.weight.detach())
        f.write(struct.pack("<f", g)); f.write(p)
        # decay
        f.write(b.ssm.decay_param.detach().cpu().numpy().astype(np.float32).tobytes())
        # out_proj
        g, p = pack_ternary_weights(b.ssm.out_proj.weight.detach())
        f.write(struct.pack("<f", g)); f.write(p)
        # norm2
        f.write(b.norm2.weight.detach().cpu().numpy().astype(np.float32).tobytes())
        # ffn_in, ffn_out
        g, p = pack_ternary_weights(b.ffn_in.weight.detach())
        f.write(struct.pack("<f", g)); f.write(p)
        g, p = pack_ternary_weights(b.ffn_out.weight.detach())
        f.write(struct.pack("<f", g)); f.write(p)

    # Final norm & head
    f.write(model.norm_f.weight.detach().cpu().numpy().astype(np.float32).tobytes())
    g, p = pack_ternary_weights(model.head.weight.detach())
    f.write(struct.pack("<f", g)); f.write(p)

# Export Vocab
vocab = tokenizer.get_vocab()
with open("vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False)

size_mb = os.path.getsize(bin_path) / 1024 / 1024
print(f"🎉 Export Complete!")
print(f"   File Name: {bin_path}")
print(f"   File Size: {size_mb:.2f} MB (Fits entirely in CPU L3 Cache!)")

# Google Colab 自動ダウンロード
try:
    from google.colab import files
    print("\n⬇️ Triggering Browser Download...")
    files.download(bin_path)
    files.download("vocab.json")
except Exception as e:
    print(f"Note: If not running inside Google Colab, download manually from filesystem.")